# Скачивание трёх моделей в DataSphere — Python 3.14

Скачивает SocraticLM, TutorRL-7B и `ml4ed-epfl/TR-finetuned-adversary`.
Для скачивания нужны только `huggingface_hub`, доступ к Hugging Face и место на диске.
PyTorch, transformers, peft и GPU не нужны и здесь не устанавливаются.

Если первые две модели уже скачаны, выполните настройки, установку библиотеки,
общие функции и блок атакующего ученика. Повторная загрузка использует кеш HF.
Не обновляйте этот ноутбук поверх запущенного эксперимента: используйте отдельное ядро.
Запуск диалогов выполняется в `2models_student_judge_fixed.ipynb`; его GPU-зависимости
и совместимость с Python 3.14 этим ноутбуком не проверяются.

In [ ]:
# Вариант А: хранилище проекта (после увеличения до 50 ГБ)
MODELS_ROOT = "/home/jupyter/datasphere/project/models"
# Вариант Б: файловое хранилище. Вставьте путь из Copy path (без /models на конце)
# MODELS_ROOT = "<путь_к_файловому_хранилищу>"

# --- SocraticLM ---
SOCRATICLM_DIR = MODELS_ROOT + "/SocraticLM"
# Открытая копия: файлы побайтно совпадают с официальным CogBase-USTC/SocraticLM, токен не нужен
SOCRATICLM_REPO_ID = "CogBase-USTC/Qwen2.5-Math-7B-Instruct-SocraticLM"
# Официальный репозиторий с закрытым доступом (нужен секрет HFTOKEN, см. шаг 3)
# SOCRATICLM_REPO_ID = "CogBase-USTC/SocraticLM"

# --- TutorRL ---
TUTORRL_DIR = MODELS_ROOT + "/TutorRL-7B"
TUTORRL_REPO_ID = "eth-nlped/TutorRL-7B"  # публичный, токен не нужен

In [ ]:
import os, sys, shutil, json, fnmatch, hashlib
from pathlib import Path
import subprocess

print("Python:", sys.version)
Path(MODELS_ROOT).mkdir(parents=True, exist_ok=True)
print("Свободно GB:", round(shutil.disk_usage(MODELS_ROOT).free / 1e9, 2))

In [ ]:
# Используем pip именно текущего ядра; не собираем зависимости из исходников.
subprocess.run([sys.executable, "-m", "pip", "install", "--only-binary=:all:",
                "huggingface-hub==2.2.0"], check=True)
# Если huggingface_hub уже был импортирован до обновления, перезапустите ядро.

In [ ]:
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

if version("huggingface-hub") != "2.2.0":
    raise RuntimeError("Установка не завершилась: нужна huggingface-hub 2.2.0")
hf_token = os.environ.get("HFTOKEN") or os.environ.get("HF_TOKEN")
api = HfApi(token=hf_token)
patterns = ["*.json", "*.safetensors", "*.model", "*.txt", "*.tiktoken", "*.jinja", "README.md"]


def repo_info(repo_id, revision=None):
    return api.model_info(repo_id, revision=revision, files_metadata=True)


def selected_files(info):
    return [f for f in info.siblings if any(fnmatch.fnmatch(f.rfilename, p) for p in patterns)]


def download_checked(info, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    files = selected_files(info)
    if not files or any(f.size is None for f in files):
        raise RuntimeError("Не удалось определить размеры файлов модели")
    missing_bytes = sum(f.size for f in files
                        if not (destination / f.rfilename).is_file()
                        or (destination / f.rfilename).stat().st_size != f.size)
    free = shutil.disk_usage(destination).free
    print(f"{info.id}: скачать до {missing_bytes/1e9:.2f} GB, свободно {free/1e9:.2f} GB")
    if free < missing_bytes + 2_000_000_000:
        raise RuntimeError("Недостаточно места для модели и запаса 2 GB")
    snapshot_download(repo_id=info.id, revision=info.sha, local_dir=str(destination),
                      token=hf_token, allow_patterns=patterns, max_workers=4)
    for entry in files:
        path = destination / entry.rfilename
        if not path.is_file() or path.stat().st_size != entry.size:
            raise RuntimeError(f"Отсутствует или неполон файл: {entry.rfilename}")
        # Для LFS-файлов проверяем SHA256 из метаданных HF, не только размер.
        lfs = entry.lfs
        expected = lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
        if expected:
            with path.open("rb") as stream:
                digest = hashlib.file_digest(stream, "sha256").hexdigest()
            if digest != expected:
                raise RuntimeError(f"Не совпал SHA256: {entry.rfilename}")
    return str(destination.resolve())


def verify_weights(directory):
    directory = Path(directory)
    indices = list(directory.glob("*.safetensors.index.json"))
    if indices:
        for index in indices:
            data = json.loads(index.read_text(encoding="utf-8"))
            shards = set(data["weight_map"].values())
            missing = [name for name in shards if not (directory / name).is_file()]
            if missing:
                raise RuntimeError(f"Не хватает частей весов: {missing}")
    elif not list(directory.glob("*.safetensors")):
        raise RuntimeError(f"Нет весов safetensors: {directory}")
    print("Веса проверены:", directory)


def download_model(repo_id, destination):
    pin_path = Path(destination) / "download_revision.json"
    saved = json.loads(pin_path.read_text(encoding="utf-8")) if pin_path.exists() else None
    if saved and saved["repo_id"] != repo_id:
        raise ValueError("Каталог уже используется для другой модели")
    info = repo_info(repo_id, saved["revision"] if saved else None)
    download_checked(info, destination)
    verify_weights(destination)
    pin_path.write_text(json.dumps({"repo_id": repo_id, "revision": info.sha}, indent=2), encoding="utf-8")

## SocraticLM — пропустите, если уже скачана

In [ ]:
download_model(SOCRATICLM_REPO_ID, SOCRATICLM_DIR)

## TutorRL-7B — пропустите, если уже скачана

In [ ]:
download_model(TUTORRL_REPO_ID, TUTORRL_DIR)

## Атакующий ученик

Ревизия фиксируется в `adversary_manifest.json`. Если это LoRA-адаптер, дополнительно
скачивается его базовая модель. Токен берётся из `HFTOKEN` / `HF_TOKEN`; значения не печатаются.

In [ ]:
ADVERSARY_REPO_ID = "ml4ed-epfl/TR-finetuned-adversary"
ADVERSARY_DIR = Path(MODELS_ROOT) / "TR-finetuned-adversary"
MANIFEST_PATH = Path(MODELS_ROOT) / "adversary_manifest.json"
saved = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else None
if saved and saved["repo_id"] != ADVERSARY_REPO_ID:
    raise ValueError("Manifest относится к другой модели")
info = repo_info(ADVERSARY_REPO_ID, saved["revision"] if saved else None)
files = {f.rfilename for f in info.siblings}
is_adapter = "adapter_config.json" in files
if not any(f.endswith(".safetensors") for f in files):
    raise RuntimeError("В репозитории нет safetensors; формат нужно проверить отдельно")
manifest = {"repo_id": ADVERSARY_REPO_ID, "revision": info.sha,
            "kind": "adapter" if is_adapter else "full",
            "model_dir": str(ADVERSARY_DIR.resolve()), "base_dir": None}
if is_adapter:
    config_path = hf_hub_download(ADVERSARY_REPO_ID, "adapter_config.json",
                                  revision=info.sha, token=hf_token)
    adapter_config = json.loads(Path(config_path).read_text(encoding="utf-8"))
    base_id = adapter_config.get("base_model_name_or_path")
    if not base_id or base_id.startswith(("/", ".")):
        raise RuntimeError("У адаптера нет доступного HF ID базовой модели")
    base_revision = saved.get("base_revision") if saved else adapter_config.get("revision")
    base_info = repo_info(base_id, base_revision)
    base_dir = Path(MODELS_ROOT) / "TR-adversary-base"
    manifest.update(base_repo_id=base_id, base_revision=base_info.sha,
                    base_dir=download_checked(base_info, base_dir))
manifest["model_dir"] = download_checked(info, ADVERSARY_DIR)
verify_weights(ADVERSARY_DIR)
if manifest["base_dir"]:
    verify_weights(manifest["base_dir"])
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Готово. Manifest:", MANIFEST_PATH)

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
config_dir = Path(manifest["base_dir"] or manifest["model_dir"])
config = json.loads((config_dir / "config.json").read_text(encoding="utf-8"))
print("Тип модели:", config.get("model_type"))
print("Архитектуры:", config.get("architectures"))
print("Контекст:", config.get("max_position_embeddings"))
print("Формат весов:", manifest["kind"])
print("Manifest:", MANIFEST_PATH)

## После скачивания

Веса и manifest сохранены в прежние каталоги, совместимые с ноутбуком эксперимента.
Эта проверка подтверждает наличие файлов, а не загрузку модели на GPU.

- При 401/403 проверьте доступ к репозиторию и принятые условия на Hugging Face.
- При сетевой ошибке проверьте интернет/NAT DataSphere, затем повторите ячейку.
- При нехватке места увеличьте хранилище; если ученик — адаптер, место нужно и под базу.
- Для установки пакета нужны доступ к pypi.org и files.pythonhosted.org.